In [1]:
import os
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

load_dotenv()


True

In [2]:

LLM_PROVIDER_URL = os.getenv("LLM_PROVIDER_URL")
LLM_API_KEY = os.getenv("LLM_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL", "gemini-2.5-flash")
LLM_MAX_TOKENS = int(os.getenv("LLM_MAX_TOKENS", 6096))
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", 0.4))
LLM_TIMEOUT = int(os.getenv("LLM_TIMEOUT", 60))


In [3]:

# 2. Define a dummy tool
@tool
def get_weather(location: str) -> str:
    """Get the current weather forecast for a given location."""
    mock_data = {
        "Chennai": "32°C, Partly Cloudy with high humidity",
        "New York": "18°C, Clear",
        "Tokyo": "22°C, Light Rain"
    }
    return mock_data.get(location, f"25°C and sunny in {location}")


tools = [get_weather]



In [4]:

# 3. Define Graph State
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# 4. Initialize ChatGoogleGenerativeAI using Gemini OpenAI-compatible endpoint
llm = ChatGoogleGenerativeAI(
            model=LLM_MODEL,
            google_api_key=LLM_API_KEY,
            temperature=LLM_TEMPERATURE,
            max_tokens=LLM_MAX_TOKENS,
            timeout=LLM_TIMEOUT,
        )
llm_with_tools = llm.bind_tools(tools)



In [5]:

# 5. Define Agent Node
def call_agent(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


In [6]:


# 6. Build the Graph
workflow = StateGraph(AgentState)

workflow.add_node("agent", call_agent)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

app = workflow.compile()



In [8]:
if __name__ == "__main__":
    query = "What is the weather like in Chennai right now?"
    initial_input = {"messages": [HumanMessage(content=query)]}

    for chunk in app.stream(initial_input, stream_mode="values"):
        message = chunk["messages"][-1]
        print(f"\n--- {message.__class__.__name__} ---")
        if message.content:
            print(message.content)
        if getattr(message, "tool_calls", None):
            print(f"Tool calls: {message.tool_calls}")


--- HumanMessage ---
What is the weather like in Chennai right now?

--- AIMessage ---
Tool calls: [{'name': 'get_weather', 'args': {'location': 'Chennai'}, 'id': 'call_174313', 'type': 'tool_call'}]

--- ToolMessage ---
32°C, Partly Cloudy with high humidity

--- AIMessage ---
[{'type': 'text', 'text': 'The weather in Chennai right now is 32°C and partly cloudy with high humidity.', 'extras': {'signature': 'EmgKZgERTTIPOY1Hq+zAFqAzGLzv132KOfzkLRkOrr0s1wgB6D8stpnFzzHb6v07cuwv+b0eta7JIXxWxNPtWg974KFvY8oiBFPMvvSu6uC6KvB3ct4juzUKNvLC/8NEUrDxcUYroJep2A=='}}]
